# 10 — Exports

Produces all static artifacts consumed by the front-end visualization layer.
Run after `09_modeling.ipynb` — all exports depend on processed + modelling outputs.

| Export | Format | Destination | Consumer |
|---|---|---|---|
| `airports.json` | JSON | `data/exports/` | Map markers, dashboard cards |
| `routes.geojson` | GeoJSON | `data/exports/` | Arc map / globe (domestic) |
| `airport_summary.json` | JSON | `data/exports/` | Dashboard stat cards |
| `heatmaps.json` + HTML | JSON + HTML | `data/exports/` | Hour × month delay chart |
| `model_summary.json` | JSON | `data/exports/` | Blog model-results section |
| `intl_routes.json` | JSON | `data/exports/` | Globe international arc layer |

In [1]:
import json
import sys
import warnings
import numpy as np
import pandas as pd
import plotly.express as px
from pathlib import Path

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))
from config import PROCESSED_DATA_PATH, EXPORTS_PATH

PROC    = PROJECT_ROOT / PROCESSED_DATA_PATH
EXPORTS = PROJECT_ROOT / EXPORTS_PATH
EXPORTS.mkdir(parents=True, exist_ok=True)

AIRPORTS = ['IAD', 'DCA', 'BWI']

# Load processed data
feat = pd.read_parquet(PROC / 'flights_featured.parquet')
feat['FlightDate'] = pd.to_datetime(feat['FlightDate'])
operated = feat[feat['Cancelled'] == 0].copy()

# Load modelling exports
route_clusters    = pd.read_parquet(EXPORTS / 'route_clusters.parquet')
prophet_forecasts = pd.read_parquet(EXPORTS / 'prophet_forecasts.parquet')
anomalous_routes  = pd.read_parquet(EXPORTS / 'anomalous_routes.parquet')
gam_partials      = pd.read_parquet(EXPORTS / 'gam_partials.parquet')

print(f'feat:           {len(feat):,} rows')
print(f'route_clusters: {len(route_clusters)} routes')
print(f'Exports dir:    {EXPORTS}')

feat:           2,823,044 rows
route_clusters: 256 routes
Exports dir:    C:\Users\kabec\Documents\three_airports\data\exports


## 1. Airport lat/lon lookup

In [2]:
AIRPORT_COORDS = {
    # DMV hubs
    'IAD': (38.9531, -77.4565), 'DCA': (38.8512, -77.0402), 'BWI': (39.1754, -76.6683),
    # Northeast
    'BOS': (42.3656, -71.0096), 'JFK': (40.6413, -73.7781), 'LGA': (40.7769, -73.8740),
    'EWR': (40.6895, -74.1745), 'HPN': (41.0670, -73.7076), 'ISP': (40.7952, -73.1002),
    'PVD': (41.7226, -71.4282), 'BDL': (41.9389, -72.6832), 'ALB': (42.7483, -73.8017),
    'SYR': (43.1112, -76.1063), 'ROC': (43.1189, -77.6724), 'BUF': (42.9405, -78.7322),
    'PWM': (43.6462, -70.3093), 'MHT': (42.9326, -71.4357), 'BTV': (44.4720, -73.1533),
    'PBG': (44.6509, -73.4681), 'OGS': (44.6819, -75.4655),
    # Mid-Atlantic / Southeast
    'ORF': (36.8976, -76.0132), 'RIC': (37.5052, -77.3197), 'RDU': (35.8776, -78.7875),
    'CLT': (35.2140, -80.9431), 'ATL': (33.6407, -84.4277), 'MIA': (25.7959, -80.2870),
    'FLL': (26.0726, -80.1527), 'PBI': (26.6832, -80.0956), 'TPA': (27.9755, -82.5332),
    'RSW': (26.5362, -81.7552), 'MCO': (28.4312, -81.3081), 'JAX': (30.4941, -81.6879),
    'SAV': (32.1276, -81.2021), 'CHS': (32.8986, -80.0405), 'ILM': (34.2706, -77.9026),
    'MYR': (33.6797, -78.9283), 'GSP': (34.8957, -82.2189), 'AVL': (35.4362, -82.5418),
    'GSO': (36.0978, -79.9373), 'FAY': (34.9912, -78.8802), 'SJU': (18.4374, -66.0018),
    # Appalachia / EAS
    'CKB': (39.2966, -80.2282), 'JST': (40.3162, -78.8336),
    'SHD': (38.2638, -78.8964), 'HTS': (38.3667, -82.5580),
    # Southeast interior
    'BNA': (36.1245, -86.6782), 'MEM': (35.0424, -89.9767), 'BHM': (33.5629, -86.7535),
    'HSV': (34.6372, -86.7751), 'MOB': (30.6912, -88.2428), 'CHA': (35.0353, -85.2036),
    'TYS': (35.8110, -83.9940), 'SDF': (38.1744, -85.7360), 'LEX': (38.0365, -84.6060),
    # Midwest
    'ORD': (41.9742, -87.9073), 'MDW': (41.7868, -87.7522), 'STL': (38.7487, -90.3700),
    'MCI': (39.2976, -94.7139), 'DSM': (41.5340, -93.6631), 'OMA': (41.3032, -95.8941),
    'MSP': (44.8820, -93.2218), 'MKE': (42.9472, -87.8966), 'GRR': (42.8808, -85.5228),
    'IND': (39.7173, -86.2944), 'CMH': (39.9980, -82.8919), 'CLE': (41.4117, -81.8498),
    'PIT': (40.4915, -80.2329), 'DTW': (42.2162, -83.3554), 'DAY': (39.9024, -84.2194),
    'CVG': (39.0488, -84.6678), 'CID': (41.8847, -91.7108),
    # South / Texas
    'DFW': (32.8998, -97.0403), 'DAL': (32.8471, -96.8518), 'IAH': (29.9902, -95.3368),
    'HOU': (29.6454, -95.2789), 'SAT': (29.5337, -98.4698), 'AUS': (30.1975, -97.6664),
    'MSY': (29.9934, -90.2580), 'LIT': (34.7294, -92.2243), 'OKC': (35.3931, -97.6007),
    'TUL': (36.1984, -95.8881), 'ELP': (31.8072, -106.3779),
    # Mountain / Southwest
    'DEN': (39.8561, -104.6737), 'SLC': (40.7884, -111.9778), 'ABQ': (35.0402, -106.6090),
    'PHX': (33.4373, -112.0078), 'LAS': (36.0840, -115.1537), 'TUS': (32.1161, -110.9410),
    # West Coast
    'LAX': (33.9425, -118.4081), 'SFO': (37.6213, -122.3790), 'SEA': (47.4502, -122.3088),
    'PDX': (45.5898, -122.5951), 'SAN': (32.7336, -117.1897), 'SJC': (37.3626, -121.9290),
    'OAK': (37.7213, -122.2208), 'SMF': (38.6954, -121.5908), 'BUR': (34.2007, -118.3585),
    'SNA': (33.6757, -117.8682), 'ONT': (34.0560, -117.6012),
    # Hawaii / Alaska
    'HNL': (21.3187, -157.9224), 'OGG': (20.8986, -156.4305), 'ANC': (61.1741, -149.9961),
}

# Confirm all DMV home airports are present
for ap in AIRPORTS:
    lat, lon = AIRPORT_COORDS[ap]
    print(f'{ap}: {lat:.4f}, {lon:.4f}')

# Count how many route destinations we can locate
all_dests = route_clusters['Dest'].unique()
missing   = [d for d in all_dests if d not in AIRPORT_COORDS]
print(f'\nRoute destinations: {len(all_dests)}  |  Missing coords: {len(missing)}')
if missing:
    print(f'  Missing: {sorted(missing)}')

out = EXPORTS / 'airports.json'
json.dump(
    {k: {'lat': v[0], 'lon': v[1]} for k, v in AIRPORT_COORDS.items()},
    open(out, 'w'), indent=2
)
print(f'\nSaved: {out}')

IAD: 38.9531, -77.4565
DCA: 38.8512, -77.0402
BWI: 39.1754, -76.6683

Route destinations: 118  |  Missing coords: 28
  Missing: ['ACK', 'AGS', 'BGR', 'BTR', 'CAE', 'CAK', 'COS', 'CRW', 'ECP', 'EYW', 'FNT', 'HHH', 'ICT', 'JAN', 'LAN', 'LWB', 'MGM', 'MSN', 'MVY', 'PHL', 'PNS', 'SBN', 'SRQ', 'STT', 'TLH', 'TVC', 'VPS', 'XNA']

Saved: C:\Users\kabec\Documents\three_airports\data\exports\airports.json


## 2. Route arcs — GeoJSON

In [3]:
import csv, urllib.request

# Download OurAirports municipality names for destination city labels
_url = 'https://davidmegginson.github.io/ourairports-data/airports.csv'
_content = urllib.request.urlopen(_url, timeout=30).read().decode('utf-8')
oa_cities = {}
for row in csv.DictReader(_content.splitlines()):
    iata = row.get('iata_code', '').strip()
    if iata:
        city = row.get('municipality', '').strip()
        if city:
            oa_cities[iata] = city
print(f'OurAirports city names loaded: {len(oa_cities):,}')

features = []
skipped  = 0

for _, row in route_clusters.iterrows():
    orig, dest = row['Origin'], row['Dest']
    if orig not in AIRPORT_COORDS or dest not in AIRPORT_COORDS:
        skipped += 1
        continue

    o_lat, o_lon = AIRPORT_COORDS[orig]
    d_lat, d_lon = AIRPORT_COORDS[dest]

    props = {
        'origin':        orig,
        'dest':          dest,
        'dest_city':     oa_cities.get(dest, ''),
        'late_rate':     round(float(row['late_rate']), 4),
        'flight_freq':   int(row['flight_freq']),
        'mean_delay':    round(float(row['mean_arr_delay']), 2),
        'mean_fare':     round(float(row['mean_fare']), 2),
        'mean_distance': round(float(row['mean_distance']), 1),
        'carrier_share': round(float(row['carrier_share']), 4),
        'nas_share':     round(float(row['nas_share']), 4),
        'cancel_rate':   round(float(row['cancel_rate']), 4),
        'hdbscan':       int(row['cluster']),
        'kmeans':        int(row['kmeans']),
    }

    features.append({
        'type': 'Feature',
        'geometry': {
            'type': 'LineString',
            'coordinates': [[o_lon, o_lat], [d_lon, d_lat]],
        },
        'properties': props,
    })

geojson = {'type': 'FeatureCollection', 'features': features}
out = EXPORTS / 'routes.geojson'
with open(out, 'w') as f:
    json.dump(geojson, f)

print(f'Routes exported: {len(features)}  |  Skipped (missing coords): {skipped}')
print(f'Saved: {out}  ({out.stat().st_size / 1e3:.1f} KB)')
# Spot-check a few city lookups
for ap in ['BOS', 'LAX', 'ORD', 'ATL', 'DEN']:
    print(f'  {ap}: {oa_cities.get(ap, "(missing)")}')


OurAirports city names loaded: 8,694
Routes exported: 217  |  Skipped (missing coords): 39
Saved: C:\Users\kabec\Documents\three_airports\data\exports\routes.geojson  (85.7 KB)
  BOS: Boston
  LAX: Los Angeles
  ORD: Chicago
  ATL: Atlanta
  DEN: Denver


## 3. Model Summary JSON

Key numbers from all four model families — single source of truth for blog copy.

In [4]:
# Prophet gaps (late_rate: actual - forecast, post-2022)
gap = (
    prophet_forecasts[prophet_forecasts['target'] == 'late_rate']
    .assign(gap=lambda x: x['y'] - x['yhat'],
            year=lambda x: x['ds'].dt.year)
    .dropna(subset=['gap'])
    .groupby(['Origin', 'year'])['gap']
    .mean()
    .reset_index()
)

prophet_summary = {}
for ap in AIRPORTS:
    sub      = gap[(gap['Origin'] == ap) & (gap['year'] >= 2022)]
    avg_gap  = float(sub['gap'].mean())
    worst    = sub.loc[sub['gap'].idxmax()]
    prophet_summary[ap] = {
        'avg_gap_2022_2025':    round(avg_gap, 4),
        'worst_year':           int(worst['year']),
        'worst_gap':            round(float(worst['gap']), 4),
        'by_year':              {int(r['year']): round(float(r['gap']), 4)
                                 for _, r in sub.iterrows()},
    }

# HDBSCAN cluster counts
hdb_counts = route_clusters['cluster'].value_counts().sort_index().to_dict()
noise_n    = int(hdb_counts.get(-1, 0))
hdbscan_summary = {
    'n_clusters':  len(hdb_counts) - (1 if -1 in hdb_counts else 0),
    'noise_count': noise_n,
    'noise_pct':   round(noise_n / len(route_clusters) * 100, 1),
    'cluster_sizes': {int(k): int(v) for k, v in hdb_counts.items()},
}

# K-means cluster counts (Cluster 0 routes only)
km_counts = route_clusters[route_clusters['kmeans'] >= 0]['kmeans'].value_counts().sort_index().to_dict()
kmeans_summary = {
    'best_k':         len(km_counts),
    'silhouette':     0.2552,
    'cluster_sizes':  {int(k): int(v) for k, v in km_counts.items()},
}

# Isolation Forest persistent anomalies
persistent = (
    anomalous_routes[anomalous_routes['year'] >= 2022]
    .groupby(['Origin', 'Dest'])['is_anomalous']
    .sum()
    .reset_index()
    .query('is_anomalous >= 2')
    .sort_values('is_anomalous', ascending=False)
)
top = persistent.iloc[0] if len(persistent) > 0 else None
if_summary = {
    'total_persistent': int(len(persistent)),
    'by_airport':       {ap: int(len(persistent[persistent['Origin'] == ap])) for ap in AIRPORTS},
    'most_anomalous':   f'{top["Origin"]}-{top["Dest"]}' if top is not None else None,
    'most_anomalous_years': int(top['is_anomalous']) if top is not None else None,
}

model_summary = {
    'prophet':         prophet_summary,
    'hdbscan':         hdbscan_summary,
    'kmeans':          kmeans_summary,
    'isolation_forest': if_summary,
    'gam':             {'auc': 0.674, 'note': 'wx_had_fog not significant (p=0.88)'},
}

out = EXPORTS / 'model_summary.json'
with open(out, 'w') as f:
    json.dump(model_summary, f, indent=2)

print(json.dumps(model_summary, indent=2))
print(f'\nSaved: {out}')

{
  "prophet": {
    "IAD": {
      "avg_gap_2022_2025": -0.0013,
      "worst_year": 2025,
      "worst_gap": 0.0231,
      "by_year": {
        "2022": 0.0083,
        "2023": -0.0054,
        "2024": -0.0056,
        "2025": 0.0231,
        "2026": -0.027
      }
    },
    "DCA": {
      "avg_gap_2022_2025": -0.0069,
      "worst_year": 2025,
      "worst_gap": 0.0338,
      "by_year": {
        "2022": 0.0079,
        "2023": -0.0453,
        "2024": -0.0295,
        "2025": 0.0338,
        "2026": -0.0015
      }
    },
    "BWI": {
      "avg_gap_2022_2025": 0.0395,
      "worst_year": 2022,
      "worst_gap": 0.0889,
      "by_year": {
        "2022": 0.0889,
        "2023": 0.0593,
        "2024": 0.0484,
        "2025": 0.027,
        "2026": -0.0259
      }
    }
  },
  "hdbscan": {
    "n_clusters": 2,
    "noise_count": 28,
    "noise_pct": 10.9,
    "cluster_sizes": {
      "-1": 28,
      "0": 223,
      "1": 5
    }
  },
  "kmeans": {
    "best_k": 6,
    "silhouette": 

## 4. Airport summary JSON

In [5]:
summary = {}
for ap in AIRPORTS:
    ap_feat = feat[feat['Origin'] == ap]
    ap_op   = operated[operated['Origin'] == ap]

    # Volume
    total_flights  = int(len(ap_feat))
    total_operated = int(len(ap_op))
    total_cancelled = int(ap_feat['Cancelled'].sum())

    # Delay metrics (operated only)
    late_rate    = float(ap_op['is_late'].mean())
    mean_delay   = float(ap_op['ArrDelayMinutes'].mean())
    cancel_rate  = float(ap_feat['Cancelled'].mean())

    # By year (for sparklines)
    by_year = (
        ap_feat.assign(year=ap_feat['FlightDate'].dt.year)
        .groupby('year')
        .agg(
            flights    = ('Cancelled', 'count'),
            late_rate  = ('is_late',   'mean'),
            cancel_rate= ('Cancelled', 'mean'),
        )
        .round(4)
        .reset_index()
    )

    # Top 5 routes by volume
    top_routes = (
        ap_op.groupby('Dest')
        .agg(flights=('is_late','count'), late_rate=('is_late','mean'))
        .nlargest(5, 'flights')
        .reset_index()
    )

    # Anomalous routes count
    anomalous_n = int(len(
        anomalous_routes[
            (anomalous_routes['Origin'] == ap) &
            (anomalous_routes['year'] >= 2022) &
            (anomalous_routes['is_anomalous'] == 1)
        ]['Dest'].unique()
    ))

    summary[ap] = {
        'total_flights':   total_flights,
        'total_operated':  total_operated,
        'total_cancelled': total_cancelled,
        'late_rate':       round(late_rate, 4),
        'mean_delay_min':  round(mean_delay, 2),
        'cancel_rate':     round(cancel_rate, 4),
        'anomalous_routes_post2022': anomalous_n,
        'by_year': by_year.to_dict(orient='records'),
        'top_routes': top_routes.assign(
            late_rate=top_routes['late_rate'].round(4)
        ).to_dict(orient='records'),
        'lat': AIRPORT_COORDS[ap][0],
        'lon': AIRPORT_COORDS[ap][1],
    }
    print(f'{ap}: {total_flights:,} flights  late={late_rate:.3f}  cancel={cancel_rate:.3f}  anomalous_routes={anomalous_n}')

out = EXPORTS / 'airport_summary.json'
with open(out, 'w') as f:
    json.dump(summary, f, indent=2)
print(f'\nSaved: {out}')

IAD: 559,793 flights  late=0.173  cancel=0.018  anomalous_routes=27


DCA: 1,229,075 flights  late=0.190  cancel=0.029  anomalous_routes=36


BWI: 1,034,176 flights  late=0.201  cancel=0.023  anomalous_routes=56

Saved: C:\Users\kabec\Documents\three_airports\data\exports\airport_summary.json


## 5. Delay heatmaps — Plotly HTML

In [6]:
heatmaps = {}
for ap in AIRPORTS:
    ap_op = operated[operated['Origin'] == ap].copy()
    ap_op['hour']  = (ap_op['CRSDepTime'] // 100).clip(0, 23)
    ap_op['month'] = ap_op['FlightDate'].dt.month

    matrix = (
        ap_op.groupby(['hour', 'month'])['is_late']
        .mean()
        .unstack(fill_value=None)   # columns = months 1-12, index = hours 0-23
        .reindex(index=range(24), columns=range(1, 13))
    )

    # Plotly heatmap HTML
    fig = px.imshow(
        matrix,
        labels=dict(x='Month', y='Hour of Day', color='Late Rate'),
        x=[str(m) for m in range(1, 13)],
        y=[str(h) for h in range(24)],
        color_continuous_scale='RdYlGn_r',
        zmin=0, zmax=0.4,
        title=f'{ap} — Delay Rate by Departure Hour × Month (2015–2025)',
        aspect='auto',
    )
    fig.update_layout(margin=dict(l=40, r=20, t=50, b=40))
    out_html = EXPORTS / f'heatmap_{ap}.html'
    fig.write_html(str(out_html))

    # JSON version for custom frontend charts
    heatmaps[ap] = {
        'hours':  list(range(24)),
        'months': list(range(1, 13)),
        'values': [
            [round(float(matrix.loc[h, m]), 4) if pd.notna(matrix.loc[h, m]) else None
             for m in range(1, 13)]
            for h in range(24)
        ],
    }
    print(f'{ap}: saved heatmap_{ap}.html')

out_json = EXPORTS / 'heatmaps.json'
with open(out_json, 'w') as f:
    json.dump(heatmaps, f)
print(f'\nSaved: {out_json}')

IAD: saved heatmap_IAD.html


DCA: saved heatmap_DCA.html


BWI: saved heatmap_BWI.html

Saved: C:\Users\kabec\Documents\three_airports\data\exports\heatmaps.json


## 6. International routes — Globe.gl arc layer

Aggregates `t100_intl_dmv.parquet` to one row per DMV-origin → foreign-dest pair.
Airport coords sourced from the OurAirports public CSV (9 000+ IATA codes).
Routes with missing coords (defunct/military airports, <400 total passengers) are skipped.

In [7]:
import csv
import urllib.request

# Download OurAirports coords (free, ~9k IATA codes)
_url = 'https://davidmegginson.github.io/ourairports-data/airports.csv'
_content = urllib.request.urlopen(_url, timeout=30).read().decode('utf-8')
oa_coords = {}
for row in csv.DictReader(_content.splitlines()):
    iata = row.get('iata_code', '').strip()
    if not iata:
        continue
    try:
        oa_coords[iata] = (float(row['latitude_deg']), float(row['longitude_deg']))
    except (ValueError, KeyError):
        pass

print(f'OurAirports: {len(oa_coords):,} IATA codes loaded')

# Load international T100
intl_raw = pd.read_parquet(PROC / 't100_intl_dmv.parquet')

# T100 only covers US-carrier filings, so routes operated by foreign carriers
# (e.g. IAD->TSA on EVA Air) appear as inbound records (TSA->IAD) or not at all.
# Normalize both directions so the DMV airport is always the "home" end of the arc.
dmv_set = set(AIRPORTS)

def normalize(df):
    out = df[df['ORIGIN'].isin(dmv_set) & ~df['DEST'].isin(dmv_set)].copy()
    out = out.rename(columns={
        'ORIGIN': 'dmv_ap', 'DEST': 'intl_ap',
        'ORIGIN_CITY_NAME': 'dmv_city', 'DEST_CITY_NAME': 'intl_city',
        'DEST_COUNTRY_NAME': 'intl_country',
    })
    inb = df[df['DEST'].isin(dmv_set) & ~df['ORIGIN'].isin(dmv_set)].copy()
    inb = inb.rename(columns={
        'DEST': 'dmv_ap', 'ORIGIN': 'intl_ap',
        'DEST_CITY_NAME': 'dmv_city', 'ORIGIN_CITY_NAME': 'intl_city',
        'ORIGIN_COUNTRY_NAME': 'intl_country',
    })
    return pd.concat([out, inb], ignore_index=True)

normed = normalize(intl_raw)

# Aggregate to one row per dmv_ap + intl_ap pair
agg = (
    normed
    .groupby(['dmv_ap', 'intl_ap', 'intl_city', 'intl_country'])
    .agg(
        total_passengers = ('PASSENGERS',          'sum'),
        total_departures = ('DEPARTURES_PERFORMED', 'sum'),
        mean_distance    = ('DISTANCE',            'mean'),
    )
    .reset_index()
    .sort_values('total_passengers', ascending=False)
)

print(f'Unique DMV ↔ international route pairs: {len(agg)}')

# Build JSON records with coords, drop noise routes (< 500 total passengers)
features_intl = []
skipped_intl  = 0

for _, row in agg.iterrows():
    if row['total_passengers'] < 500:
        skipped_intl += 1
        continue
    dmv, intl = row['dmv_ap'], row['intl_ap']
    if dmv not in oa_coords or intl not in oa_coords:
        skipped_intl += 1
        continue

    d_lat, d_lon = oa_coords[dmv]
    i_lat, i_lon = oa_coords[intl]

    features_intl.append({
        'origin':           dmv,
        'dest':             intl,
        'dest_city':        row['intl_city'],
        'dest_country':     row['intl_country'],
        'total_passengers': int(row['total_passengers']),
        'total_departures': int(row['total_departures']),
        'mean_distance':    round(float(row['mean_distance']), 1),
        'o_lat': d_lat, 'o_lon': d_lon,
        'd_lat': i_lat, 'd_lon': i_lon,
    })

out = EXPORTS / 'intl_routes.json'
with open(out, 'w') as f:
    json.dump(features_intl, f, indent=2)

print(f'Exported: {len(features_intl)}  |  Skipped (noise + missing coords): {skipped_intl}')
print(f'Saved: {out}  ({out.stat().st_size / 1e3:.1f} KB)')
print()
print('Top 10 routes by passengers:')
for r in features_intl[:10]:
    print(f'  {r["origin"]} -> {r["dest"]:4s}  {r["dest_city"]:<35}  {r["total_passengers"]:>9,} pax')

OurAirports: 9,056 IATA codes loaded
Unique DMV ↔ international route pairs: 582
Exported: 153  |  Skipped (noise + missing coords): 429
Saved: C:\Users\kabec\Documents\three_airports\data\exports\intl_routes.json  (48.8 KB)

Top 10 routes by passengers:
  IAD -> LHR   London, United Kingdom               7,579,783 pax
  IAD -> FRA   Frankfurt, Germany                   5,963,284 pax
  IAD -> CDG   Paris, France                        4,501,031 pax
  IAD -> SAL   San Salvador, El Salvador            3,900,113 pax
  BWI -> CUN   Cancun, Mexico                       2,831,668 pax
  IAD -> MUC   Munich, Germany                      2,683,456 pax
  IAD -> DXB   Dubai, United Arab Emirates          2,660,097 pax
  IAD -> IST   Istanbul, Turkey                     2,587,349 pax
  IAD -> AMS   Amsterdam, Netherlands               2,504,875 pax
  IAD -> DOH   Doha, Qatar                          2,399,431 pax


## 7. Prophet forecast chart data

In [8]:
# Prophet forecast chart — yearly aggregated actual + forecast per airport
# Filter to late_rate target only
pf = prophet_forecasts[prophet_forecasts['target'] == 'late_rate'].copy()
pf['year'] = pf['ds'].dt.year

chart_data = {}
for ap in AIRPORTS:
    sub = pf[pf['Origin'] == ap].groupby('year').agg(
        y           = ('y',           'mean'),
        yhat        = ('yhat',        'mean'),
        yhat_lower  = ('yhat_lower',  'mean'),
        yhat_upper  = ('yhat_upper',  'mean'),
    ).reset_index()

    chart_data[ap] = [
        {
            'year':        int(r['year']),
            'y':           round(float(r['y']), 4)       if pd.notna(r['y'])   else None,
            'yhat':        round(float(r['yhat']), 4),
            'yhat_lower':  round(float(r['yhat_lower']), 4),
            'yhat_upper':  round(float(r['yhat_upper']), 4),
        }
        for _, r in sub.iterrows()
    ]

out = EXPORTS / 'prophet_chart.json'
with open(out, 'w') as f:
    json.dump(chart_data, f, indent=2)

print(f'Saved: {out}')
for ap in AIRPORTS:
    rows = chart_data[ap]
    print(f'  {ap}: {len(rows)} years  ({rows[0]["year"]}–{rows[-1]["year"]})')

Saved: C:\Users\kabec\Documents\three_airports\data\exports\prophet_chart.json


  IAD: 12 years  (2015–2026)
  DCA: 12 years  (2015–2026)
  BWI: 12 years  (2015–2026)


## 8. Anomalous routes JSON

In [9]:
# Persistent anomalous routes — flagged >= 2 years since 2022
# oa_cities was loaded in cell "2. Route arcs" (32df6435)
post22 = anomalous_routes[anomalous_routes['year'] >= 2022].copy()

persistent_routes = (
    post22.groupby(['Origin', 'Dest'])
    .agg(
        years_flagged = ('is_anomalous',  'sum'),
        mean_late_rate = ('late_rate',    'mean'),
        mean_delay     = ('mean_delay',   'mean'),
        n_years        = ('year',         'count'),
    )
    .reset_index()
    .query('years_flagged >= 2')
    .sort_values(['years_flagged', 'mean_late_rate'], ascending=[False, False])
)

anomaly_records = []
for _, r in persistent_routes.iterrows():
    dest_city = oa_cities.get(r['Dest'], '')
    anomaly_records.append({
        'origin':        r['Origin'],
        'dest':          r['Dest'],
        'dest_city':     dest_city,
        'years_flagged': int(r['years_flagged']),
        'n_years':       int(r['n_years']),
        'late_rate':     round(float(r['mean_late_rate']), 4),
        'mean_delay':    round(float(r['mean_delay']), 2),
    })

out = EXPORTS / 'anomalous_routes.json'
with open(out, 'w') as f:
    json.dump(anomaly_records, f, indent=2)

print(f'Exported {len(anomaly_records)} persistent anomalous routes')
print(f'Saved: {out}')
print()
print('Top 10:')
for r in anomaly_records[:10]:
    city = f' ({r["dest_city"]})' if r['dest_city'] else ''
    print(f'  {r["origin"]}→{r["dest"]}{city:<30}  flagged {r["years_flagged"]}/{r["n_years"]} yrs  late={r["late_rate"]:.1%}')

Exported 87 persistent anomalous routes
Saved: C:\Users\kabec\Documents\three_airports\data\exports\anomalous_routes.json

Top 10:
  BWI→SAT (San Antonio)                  flagged 5/5 yrs  late=25.5%
  BWI→HOU (Houston)                      flagged 5/5 yrs  late=25.1%
  BWI→MDW (Chicago)                      flagged 5/5 yrs  late=23.1%
  BWI→ORD (Chicago)                      flagged 5/5 yrs  late=20.9%
  DCA→STL (St Louis)                     flagged 5/5 yrs  late=20.7%
  IAD→SFO (San Francisco)                flagged 5/5 yrs  late=20.0%
  BWI→CLT (Charlotte)                    flagged 5/5 yrs  late=19.9%
  DCA→CLT (Charlotte)                    flagged 5/5 yrs  late=19.9%
  BWI→ATL (Atlanta)                      flagged 5/5 yrs  late=19.2%
  IAD→TPA (Tampa)                        flagged 5/5 yrs  late=18.3%


## 9. Crossfilter dashboard data

Pre-aggregated at `Origin × year × month × dep_hour` grain.
Each cell stores flight counts plus delay and distance bin distributions,
enabling full crossfilter interactions with ~10 K cells instead of 2.8 M raw rows.

In [ ]:

# 9. Crossfilter dashboard data
# Pre-aggregate to Origin × year × month × dep_hour grain (~10 K cells).
# Each cell stores counts + delay-bin and distance-bin distributions so the
# frontend can do full crossfilter interactions without the 2.8 M raw rows.

feat2 = feat.copy()
feat2['year']     = feat2['FlightDate'].dt.year
feat2['month']    = feat2['FlightDate'].dt.month
feat2['dep_hour'] = (feat2['CRSDepTime'] // 100).clip(0, 23)

DELAY_EDGES = [-np.inf, 0, 15, 30, 60, 120, np.inf]
DIST_EDGES  = [0, 300, 600, 1000, 1500, 2500, np.inf]

# labels=False → integer bin index (0-5), NaN for out-of-range / cancelled
feat2['delay_bin'] = pd.cut(feat2['ArrDelayMinutes'], bins=DELAY_EDGES, right=False, labels=False)
feat2['dist_bin']  = pd.cut(feat2['Distance'],        bins=DIST_EDGES,  right=False, labels=False)

KEY = ['Origin', 'year', 'month', 'dep_hour']

# Total counts per cell
counts = (
    feat2.groupby(KEY, observed=True)
    .agg(n=('Cancelled', 'count'), nc=('Cancelled', 'sum'), nl=('is_late', 'sum'))
    .reset_index()
)

# Delay bins — operated flights only (cancelled have NaN delay)
op2 = feat2[(feat2['Cancelled'] == 0) & feat2['delay_bin'].notna()].copy()
op2['delay_bin'] = op2['delay_bin'].astype(int)
delay_agg = (
    op2.groupby(KEY + ['delay_bin'])
    .size()
    .unstack('delay_bin', fill_value=0)
    .reindex(columns=range(6), fill_value=0)
    .rename(columns={i: f'db{i}' for i in range(6)})
    .reset_index()
)

# Distance bins — all flights (Distance always populated)
feat2_dist = feat2[feat2['dist_bin'].notna()].copy()
feat2_dist['dist_bin'] = feat2_dist['dist_bin'].astype(int)
dist_agg = (
    feat2_dist.groupby(KEY + ['dist_bin'])
    .size()
    .unstack('dist_bin', fill_value=0)
    .reindex(columns=range(6), fill_value=0)
    .rename(columns={i: f'xb{i}' for i in range(6)})
    .reset_index()
)

# Merge into one table
merged = (
    counts
    .merge(delay_agg, on=KEY, how='left')
    .merge(dist_agg,  on=KEY, how='left')
    .fillna(0)
)

# Encode: [origin_idx, year, month, dep_hour, n, nc, nl, db0..5, xb0..5]
ORIGIN_IDX = {ap: i for i, ap in enumerate(AIRPORTS)}
records = []
for _, r in merged.iterrows():
    records.append([
        ORIGIN_IDX[r['Origin']],
        int(r['year']), int(r['month']), int(r['dep_hour']),
        int(r['n']), int(r['nc']), int(r['nl']),
        *[int(r[f'db{i}']) for i in range(6)],
        *[int(r[f'xb{i}']) for i in range(6)],
    ])

out_data = {
    'airports': AIRPORTS,
    'cols': ['o','y','m','h','n','nc','nl',
             'db0','db1','db2','db3','db4','db5',
             'xb0','xb1','xb2','xb3','xb4','xb5'],
    'rows': records,
}

out = EXPORTS / 'crossfilter_data.json'
with open(out, 'w') as f:
    json.dump(out_data, f, separators=(',', ':'))

sz = out.stat().st_size
print(f'Rows:   {len(records):,}')
print(f'Saved:  {out}')
print(f'Size:   {sz/1e3:.1f} KB  (~{sz/10/1000:.0f} KB gzip estimate)')
print(f'Sample: {records[0]}')
print(f'\nColumns: {out_data["cols"]}')


In [ ]:

# 10a. Route monthly stats + carrier data
# Powers crossfilter table date-filtered view and the Airlines column.
# cols: [o=origin_idx, d=dest_idx, y=year, m=month, n=flights, lr=late_rate, md=mean_delay, mf=mean_fare]

feat_op2 = operated.copy()
feat_op2['year']  = feat_op2['FlightDate'].dt.year
feat_op2['month'] = feat_op2['FlightDate'].dt.month

# ── top-3 carriers per route (by total operated flights) ───────────
rc = (
    feat_op2
    .groupby(['Origin', 'Dest', 'Reporting_Airline'])
    .size()
    .reset_index(name='n')
    .sort_values(['Origin', 'Dest', 'n'], ascending=[True, True, False])
)
route_carriers: dict = {}
for (orig, dest), grp in rc.groupby(['Origin', 'Dest']):
    route_carriers[f'{orig}-{dest}'] = grp.head(3)['Reporting_Airline'].tolist()

# ── monthly aggregation per route ──────────────────────────────────
rm = (
    feat_op2
    .groupby(['Origin', 'Dest', 'year', 'month'])
    .agg(
        n          = ('is_late',         'count'),
        late       = ('is_late',         'sum'),
        mean_delay = ('ArrDelayMinutes', 'mean'),
        mean_fare  = ('db1b_avg_fare',   'mean'),
    )
    .reset_index()
)

all_dests = sorted(rm['Dest'].unique().tolist())
dest_idx  = {d: i for i, d in enumerate(all_dests)}

records = []
for _, r in rm.iterrows():
    md = round(float(r['mean_delay']), 1) if pd.notna(r['mean_delay']) else None
    mf = round(float(r['mean_fare']),  0) if pd.notna(r['mean_fare'])  else None
    lr = round(float(r['late']) / float(r['n']), 4) if r['n'] > 0 else 0.0
    records.append([
        ORIGIN_IDX[r['Origin']],
        dest_idx[r['Dest']],
        int(r['year']),
        int(r['month']),
        int(r['n']),
        lr, md, mf,
    ])

out = EXPORTS / 'route_monthly.json'
with open(out, 'w') as f:
    json.dump({
        'origins':  AIRPORTS,
        'dests':    all_dests,
        'carriers': route_carriers,
        'cols':     ['o', 'd', 'y', 'm', 'n', 'lr', 'md', 'mf'],
        'rows':     records,
    }, f, separators=(',', ':'))

print(f'Route monthly:  {len(records):,} rows  |  {out.stat().st_size/1e3:.1f} KB')
print(f'Unique routes:  {len(route_carriers)}')
print(f'Destinations:   {len(all_dests)}')
print(f'Saved: {out}')
print()
print('Sample carriers:')
for k, v in list(route_carriers.items())[:6]:
    print(f'  {k}: {v}')
